In [0]:

-- ===========================================================
-- SECTION 5 : PERFORMANCE OPTIMIZATION
-- ============================================================
 
-- Q15. Add indexes to speed up frequent query patterns
CREATE INDEX idx_orders_customer    ON orders(customer_id);
CREATE INDEX idx_orders_restaurant  ON orders(restaurant_id);
CREATE INDEX idx_orders_status_date ON orders(order_status, order_date);
CREATE INDEX idx_delivery_agent     ON order_delivery(agent_id);
 
-- Q16. EXPLAIN plan — check query execution before optimization

SELECT
    r.restaurant_name,
    SUM(o.order_amount) AS revenue
FROM restaurants r
JOIN orders o ON r.restaurant_id = o.restaurant_id
WHERE o.order_status = 'Delivered'
GROUP BY r.restaurant_id, r.restaurant_name;
 
-- Q17. Avoid SELECT * — always select only needed columns
-- BAD (slow):
-- SELECT * FROM orders WHERE order_status = 'Delivered';
 
-- GOOD (fast):
SELECT order_id, customer_id, restaurant_id, order_amount, order_date
FROM orders
WHERE order_status = 'Delivered';
 
-- Q18. Use EXISTS instead of IN for large datasets
-- IN (slower on large data):
-- SELECT * FROM customers WHERE customer_id IN (SELECT customer_id FROM orders);
 
-- EXISTS (faster):
SELECT customer_id, customer_name, city
FROM customers c
WHERE EXISTS (
    SELECT 1 FROM orders o
    WHERE o.customer_id = c.customer_id
    AND   o.order_status = 'Delivered'
);
 
-- Q19. Partition pruning — filter on indexed column first
SELECT
    order_id,
    order_amount,
    order_date
FROM orders
WHERE order_status = 'Delivered'       -- uses idx_orders_status_date
  AND order_date BETWEEN '2024-03-01' AND '2024-03-31';
 
-- Q20. Final Business Insight Query — Combine everything
WITH monthly_stats AS (
    SELECT
        DATE_FORMAT(o.order_date, '%Y-%m')  AS month,
        r.cuisine,
        COUNT(o.order_id)                   AS orders,
        SUM(o.order_amount)                 AS revenue,
        AVG(o.delivery_time)                AS avg_delivery_mins
    FROM orders o
    JOIN restaurants r ON o.restaurant_id = r.restaurant_id
    WHERE o.order_status = 'Delivered'
    GROUP BY month, r.cuisine
),
ranked AS (
    SELECT *,
        RANK() OVER (PARTITION BY month ORDER BY revenue DESC) AS cuisine_rank
    FROM monthly_stats
)
SELECT
    month,
    cuisine,
    orders,
    revenue,
    ROUND(avg_delivery_mins, 1) AS avg_delivery_mins,
    cuisine_rank
FROM ranked
WHERE cuisine_rank <= 3
ORDER BY month, cuisine_rank;
 